# Selecting Active Postcodes from the ONS Postcode Directory

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import gc

base = Path('/Users/pedrickr/Documents/ADR/Connectivity/Defo_Open')

# Load ONSPD
onspd = pd.read_csv(
    base / 'ONSPD_NOV_2025/Data/ONSPD_NOV_2025_UK.csv',
    low_memory=False
)

# Filter active postcodes
onspd_active = onspd.loc[onspd['doterm'].isna()].copy()
onspd_active['postcode_clean'] = onspd_active['pcd8'].str.replace(" ", "", regex=False)

print(
    f"Count of unique active postcodes: "
    f"{onspd_active['postcode_clean'].nunique()}"
)

# Convert to set for fast lookup
active_pc_set = set(onspd_active['postcode_clean'])

# Free memory early
del onspd
gc.collect()

# Load centroids (only needed columns)
pc_centroids = gpd.read_file(
    base / 'ONSPD_Online_Latest_Centroids_51478180977957889.gpkg'
)[['PCD8', 'geometry']]

print(f"Centroid postcode count: {pc_centroids['PCD8'].nunique()}")

# Clean postcode
pc_centroids['postcode_clean'] = pc_centroids['PCD8'].str.replace(" ", "", regex=False)

# Filter
active_pc_centroids = pc_centroids[
    pc_centroids['postcode_clean'].isin(active_pc_set)
]

# Drop unused columns
active_pc_centroids = active_pc_centroids.drop(columns=['PCD8'])

print(
    f"Active centroid count: {active_pc_centroids['postcode_clean'].nunique()}"
)

# Cleanup
del pc_centroids
gc.collect()